# 7. Learn and evaluate a baseline

**How far off is one fixed prediction on farms we’ve never seen?** Imagine your lab needs an MSC prediction for each sample from a new farm. Start with one number for every sample. Once the lab results are available, check one prediction yourself and then score the whole set.

**Open.** Restart R and save the supplied script as `scripts/my-07-baseline.R`. This chapter uses `outputs/checkpoints/farm-split.rds` from Chapter 6. In your script, run the named blocks in order.

## 7.1 Make one prediction before seeing the results

Run {span .script-block}`BLOCK · load-split` to restore the training and test tables. If the checkpoint is missing, complete Chapter 6’s save step first.

In [1]:
if (!file.exists("outputs/checkpoints/farm-split.rds")) stop("Run Chapter 6 first to create farm-split.rds.")
split_checkpoint <- readRDS("outputs/checkpoints/farm-split.rds")
train <- split_checkpoint$train
test <- split_checkpoint$test
farm_ids <- split_checkpoint$farm_ids
test_farms <- split_checkpoint$test_farms

Average log10 MSC across the **420 training records** to get your baseline. Run {span .script-block}`BLOCK · baseline`:

In [2]:
baseline_log10 <- mean(train$log10_msc)
baseline_log10
test$prediction_log10 <- rep(baseline_log10, nrow(test))
stopifnot(all(test$prediction_log10 == mean(train$log10_msc)))

[1] 0.7450425

Write down **0.7450 on the log10 scale**. That is the prediction for every test record. The line with `rep()` repeats the same number once per row of `test`. The training results set this number. The test results will tell you how far it misses.

```{admonition} Study data — express the prediction in CFU/mL
:class: note dropdown

The training table already has one log10 value per record. The baseline averages those log10 values. Raising 10 to that average gives the same prediction in the original units.

| Calculation | Result |
|---|---:|
| Average the 420 log10 concentrations | 0.7450 on the log10 scale |
| Convert the prediction back with $10^{0.7450}$ | 5.56 CFU/mL |
| Average the 420 original concentrations directly | 16.78 CFU/mL |

The first two rows give the same prediction in two units. The third row averages on the original scale. When you score a prediction, keep the observed and predicted values on the same scale.
```

## 7.2 Check one prediction by hand

**Study data.** Run {span .script-block}`BLOCK · first-test-record` to see the first test record’s lab result beside its prediction. Both values are on the log10 scale.

In [3]:
# [1] selects the first value in each column.
first_test_record <- data.frame(observed_log10 = test$log10_msc[1],
                                predicted_log10 = test$prediction_log10[1])
round(first_test_record, 4)

observed_log10,predicted_log10
<dbl>,<dbl>
0.8129,0.745


Subtract **observed minus predicted** on paper. Before opening the answer, decide whether the prediction was too high or too low.

```{admonition} Check your hand calculation
:class: tip dropdown

$$
e_1 \approx 0.8129 - 0.7450 = 0.0679.
$$

The residual is positive because the lab result was higher than the prediction. The displayed inputs are rounded to four decimal places. R uses their full precision when it scores all the records.
```

## 7.3 Score all 105 predictions

**Study data.** Run {span .script-block}`BLOCK · metrics`. The first line repeats your subtraction for every row in `test`. The next two lines summarize those 105 residuals using the [MAE and RMSE calculations from Chapter 4](04-small-farm-example.ipynb#id-4-4-turn-predictions-into-errors).

In [4]:
test$error_log10 <- test$log10_msc - test$prediction_log10
rmse <- sqrt(mean(test$error_log10^2))
mae <- mean(abs(test$error_log10))
data.frame(model = "Training-mean baseline", n_test = nrow(test),
           RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae)

model,n_test,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<int>,<dbl>,<dbl>
Training-mean baseline,105,0.5289231,0.409148


| In the result table | What to say |
|---|---|
| **105 test records** | All 105 records from the 21 held-out farms contribute to the score. |
| **MAE ≈ 0.409** | The average size of a miss is about 0.409 on the log10 scale, regardless of direction. |
| **RMSE ≈ 0.529** | Squaring the residuals gives larger misses more influence. Taking the square root brings the score back to the log10 scale. |

Use these scores as the reference when Chapter 10 compares predictions on the same held-out records. You can also run {span .script-block}`BLOCK · cross-check-error-metrics` to recompute both scores by dividing each sum by the record count. This check is silent when the values agree.

In [5]:
# Check the averages by adding the squared or absolute errors and dividing by the row count.
# RMSE takes the square root of that average.
rmse_check <- sqrt(sum((test$log10_msc - baseline_log10)^2) / nrow(test))
mae_check <- sum(abs(test$log10_msc - baseline_log10)) / nrow(test)
stopifnot(isTRUE(all.equal(rmse, rmse_check)))
stopifnot(isTRUE(all.equal(mae, mae_check)))

```{admonition} Invented data — what a log10 residual means
:class: note dropdown

Compare the difference in CFU/mL with the ratio in these two made-up cases.

| Predicted CFU/mL | Observed CFU/mL | Difference in CFU/mL | Observed / predicted | log10 residual |
|---:|---:|---:|---:|---:|
| 10 | 20 | 10 | 2 | about 0.301 |
| 100 | 200 | 100 | 2 | about 0.301 |

The observations are both twice their predictions. Their log10 residuals are therefore the same. For an observed concentration $a$ and a predicted concentration $p$:

$$
\log_{10}(a)-\log_{10}(p)=\log_{10}\!\left(\frac{a}{p}\right).
$$
```

```{admonition} Invented data — why counting records and counting farms differ
:class: note dropdown

Suppose farm A has one record of 10 CFU/mL and farm B has three records of 30 CFU/mL. Averaging the four records gives 25 CFU/mL. Averaging the two farm means, 10 and 30, gives 20 CFU/mL.
```

**Back to the study data.** Each record contributes equally to the score, so a farm with more records contributes more. The RMSE of 0.529 is an error on the log10 scale, with each residual calculated separately before summarizing.

## 7.4 Save the evidence and write two sentences

In RStudio, select and run the supplied blocks from {span .script-block}`BLOCK · score-quantified-records` through {span .script-block}`BLOCK · audit-save`. The first block checks how the same predictions score on records with quantified MSC. Run it even if you leave its details folded. The export uses its result.

In [6]:
# Evaluation subset only: the baseline is not refitted or chosen using this result.
quantified_test <- test[!test$left_censored, ]
quantified_rmse <- sqrt(mean(quantified_test$error_log10^2))
data.frame(evaluation = c("All eligible test records", "Quantified test records only"),
           n = c(nrow(test), nrow(quantified_test)),
           RMSE_log10_CFU_ml = c(rmse, quantified_rmse))

evaluation,n,RMSE_log10_CFU_ml
<chr>,<int>,<dbl>
All eligible test records,105,0.5289231
Quantified test records only,104,0.5062914


{span .script-block}`BLOCK · save-baseline-results`

In [7]:
dir.create("outputs/baseline", recursive = TRUE, showWarnings = FALSE)
score <- data.frame(model = "Training-mean baseline", training_mean_log10 = baseline_log10,
                    train_rows = nrow(train), test_rows = nrow(test),
                    train_farms = length(unique(train$FarmID)), test_farms = length(test_farms),
                    RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae,
                    quantified_test_n = nrow(quantified_test), quantified_test_RMSE = quantified_rmse)
write.csv(score, "outputs/baseline/scores.csv", row.names = FALSE)
split_table <- data.frame(FarmID = farm_ids,
                         split = ifelse(farm_ids %in% test_farms, "test", "train"))
write.csv(split_table, "outputs/baseline/farm-split.csv", row.names = FALSE)
write.csv(test[c("FarmID", "Sampling", "left_censored", "log10_msc", "prediction_log10", "error_log10")],
          "outputs/baseline/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/baseline/session-info.txt")
file.exists("outputs/baseline/scores.csv")

[1] TRUE

{span .script-block}`BLOCK · plot-training-counts`

In [8]:
png("outputs/baseline/training-distribution.png", width = 1200, height = 720, res = 150)
par(mar = c(4.5, 4.5, 2.5, 1), family = "sans", col.axis = "#294943", col.lab = "#294943")
hist(train$log10_msc, breaks = 18, col = "#8cc7b6", border = "white",
     main = "The baseline is learned from training farms",
     xlab = "Training log10 MSC (CFU/mL)", ylab = "Number of records")
abline(v = baseline_log10, col = "#a94c31", lwd = 3)
legend("topright", "Training mean", col = "#a94c31", lwd = 3, bty = "n")
invisible(dev.off())
cat("Saved outputs/baseline/training-distribution.png\n")

Saved outputs/baseline/training-distribution.png


{span .script-block}`BLOCK · audit-save`

In [9]:
write.csv(split_checkpoint$audit, "outputs/baseline/data-audit.csv", row.names = FALSE)
cat("All data and baseline checks passed.\n")

All data and baseline checks passed.


Open the files below from **Files → outputs → baseline**.

| File | What you can check |
|---|---|
| `scores.csv` | The fitted mean, number of test records and two scores |
| `test-predictions.csv` | The observed value, prediction and residual for each test record |
| `farm-split.csv` | Which farms went into training and which went into testing |
| `training-distribution.png` | Where the prediction falls in the training distribution. Open the saved image directly. |

```{admonition} Preview the saved plot
:class: note dropdown

![Training log10 MSC distribution with the training mean marked by a vertical line.](../outputs/baseline/training-distribution.png)
```

**Write two sentences in English.** First, report what you predicted, which records you tested on and the scores with their scale. Then say what limits the interpretation. Think about the study setting or the below-detection values described in Chapter 5.

```{admonition} Reference write-up — open after you try
:class: tip dropdown

A baseline that predicted the training mean achieved RMSE of about 0.529 and MAE of about 0.409 on the log10 CFU/mL scale across 105 records from 21 held-out farms. These scores describe prediction of recorded MSC in this study, including the stored substitutes for below-detection results.
```

**Finish.** Find the first record’s residual in `test-predictions.csv` and the two scores in `scores.csv`. You are ready to move on when you can explain how one became part of the other.

## 7.5 Optional AI check

Ask AI for code to verify your RMSE using the full table in your R session. Keep the supplied blocks intact. In your my- script, use {span .script-block}`BLOCK · baseline-live-workspace`, the comment-only block at the end.

In [10]:
# My AI RMSE check: add your returned R code below in your my- copy.
# Use all 105 rows of test.

```text
I am using base R. My object test contains 105 held-out records.
Observed log10 values are in test$log10_msc and fixed predictions
are in test$prediction_log10. Write code to print the row count and
calculate RMSE over all 105 rows. Use the existing table and
predictions as they are. I will run the code locally.
```

Read the reply before running it. Find the line that counts the rows and the line that calculates RMSE. Both columns in that calculation should come from the complete `test` table. Add the code below **My AI RMSE check** and run only your added lines. Look for **105** and an RMSE of about **0.529**.

```{admonition} Reference code for self-study
:class: tip dropdown

~~~r
nrow(test)
sqrt(mean((test$log10_msc - test$prediction_log10)^2))
~~~

If you use this route, record it as code inspection in your AI-use log. If you use live chat, record your actual prompt, reply and check.
```

**Next.** [Could group averages make a useful tree?](08-first-tree.ipynb)